In [1]:
import os
import re
import math
from tqdm import tqdm
# from google.colab import userdata
from huggingface_hub import login
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv()

/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Constants

BASE_MODEL = "Qwen/Qwen3-8B"
PROJECT_NAME = "Qwen3-8B-ruozhi_v2"
HF_USER = "franzyellow" 

# The run itself

RUN_NAME = "2025-12-16_08.43.01"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
REVISION = "b3990be77dc4364cea1d1dd40afcb85ceacec960" # or REVISION = None
FINETUNED_MODEL = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Uncomment this line if you wish to use my model
# FINETUNED_MODEL = f"ed-donner/{PROJECT_RUN_NAME}"

# Data

DATASET_NAME = f"{HF_USER}ruozhiba_punchline_ft"
# Or just use the one I've uploaded
# DATASET_NAME = "ed-donner/pricer-data"

# Hyperparameters for QLoRA

QUANT_4_BIT = False

%matplotlib inline

In [3]:
# Quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [4]:
# First load the base model and tokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

Loading checkpoint shards: 100%|██████████| 5/5 [00:08<00:00,  1.63s/it]


Memory footprint: 9784.9 MB


In [19]:
#system_prompt = (
    #"你是一个弱智吧网友。"
    #"回答时只接一句话，不要解释，不要总结，不要重复。"
    #"语气随意、反直觉、像随口一说。"
    #"不要讲道理，不要装作很专业。"
#)
prompt = ("欲望和收入不匹配这件事情你怎么看ANSWER:")

inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)

outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
    repetition_penalty=1.15,
    no_repeat_ngram_size=4
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

欲望和收入不匹配这件事情你怎么看ANSWER:有人想拉屎，但没钱买。这就是欲望与收入严重不符啊！这世界还有多少人活在梦里？醒醒吧！！！有梦想是好的，但是要先有钱才能圆梦！！！别做梦了好吗
